# Exploration notebook
This is for Phase 4 of the project.
The data pipeline is ready

In [5]:
# First disconnect from the scanner test
ib.disconnect()

In [3]:
# Wikipedia scrape
import pandas as pd
import requests

url = 'https://en.wikipedia.org/wiki/Russell_1000_Index'
headers = {'User-Agent': 'Mozilla/5.0'}
response = requests.get(url, headers=headers)

from io import StringIO

tables = pd.read_html(StringIO(response.text))
for i, t in enumerate(tables):
    print(f"Table {i}: {t.shape}, columns: {list(t.columns)[:5]}")

Table 0: (10, 2), columns: [0, 1]
Table 1: (2, 3), columns: ['Category', 'All-time highs[2]', 'All-time highs[2].1']
Table 2: (31, 3), columns: ['Year', 'Price return', 'Total return']
Table 3: (1005, 4), columns: ['Company', 'Symbol', 'GICS Sector', 'GICS Sub-Industry']
Table 4: (1, 2), columns: ['vteMajor United States stock market indices', 'vteMajor United States stock market indices.1']
Table 5: (8, 2), columns: ['vteMajor North and South American stock market indices', 'vteMajor North and South American stock market indices.1']


In [4]:
russell1000 = tables[3]
print(russell1000.head())
print(f"\n{russell1000['GICS Sector'].nunique()} sectors")
print(russell1000['GICS Sector'].value_counts())

               Company Symbol  GICS Sector         GICS Sub-Industry
0                   3M    MMM  Industrials  Industrial Conglomerates
1          A. O. Smith    AOS  Industrials         Building Products
2                 AAON   AAON  Industrials         Building Products
3  Abbott Laboratories    ABT  Health Care     Health Care Equipment
4               AbbVie   ABBV  Health Care           Pharmaceuticals

11 sectors
GICS Sector
Industrials               171
Financials                153
Information Technology    144
Consumer Discretionary    124
Health Care               107
Real Estate                63
Consumer Staples           60
Materials                  57
Communication Services     49
Utilities                  41
Energy                     36
Name: count, dtype: int64


In [11]:
# check for tickers with dots as it may mess with IB API
dot_tickers = russell1000[russell1000['Symbol'].str.contains(r'\.', na=False)]
print(dot_tickers[['Company', 'Symbol']])

                       Company  Symbol
117         Berkshire Hathaway   BRK.B
149     Brown–Forman (Class A)    BF.A
150     Brown–Forman (Class B)    BF.B
213  Clearway Energy (Class A)  CWEN.A
443            HEICO (Class A)   HEI.A
539           Lennar (Class B)   LEN.B
904          U-Haul (Series N)  UHAL.B


In [14]:
russell1000.to_csv('/Users/hugo/quant-research/config/russell1000_sectors.csv', index=False)

In [12]:
import yaml
with open("/Users/hugo/quant-research/config/equity_universes.yaml", 'r') as f:
    config = yaml.safe_load(f)
universe = 'russell1000'

tickers = config[universe]
print(tickers)
print(len(tickers))

['MMM', 'AOS', 'AAON', 'ABT', 'ABBV', 'ACHC', 'ACN', 'AYI', 'ADBE', 'ADT', 'WMS', 'AMD', 'ACM', 'AES', 'AMG', 'AFRM', 'AFL', 'AGCO', 'A', 'ADC', 'AGNC', 'AL', 'APD', 'ABNB', 'AKAM', 'ALK', 'ALB', 'ACI', 'AA', 'ARE', 'ALGN', 'ALLE', 'ALGM', 'LNT', 'ALSN', 'ALL', 'ALLY', 'ALNY', 'GOOGL', 'GOOG', 'MO', 'AMZN', 'AMCR', 'DOX', 'AMTM', 'AS', 'AEE', 'AAL', 'AEP', 'AXP', 'AFG', 'AMH', 'AIG', 'AMT', 'AWK', 'COLD', 'AMP', 'AME', 'AMGN', 'AMKR', 'APH', 'ADI', 'AU', 'NLY', 'AM', 'AR', 'AON', 'APA', 'APG', 'APLS', 'APO', 'APPF', 'AAPL', 'AIT', 'AMAT', 'APP', 'ATR', 'APTV', 'ARMK', 'ACGL', 'ADM', 'ARES', 'ANET', 'AWI', 'ARW', 'AJG', 'ASH', 'AIZ', 'AGO', 'ALAB', 'ASTS', 'T', 'ATI', 'TEAM', 'ATO', 'AUR', 'ADSK', 'ADP', 'AN', 'AZO', 'AVB', 'AVTR', 'AVY', 'CAR', 'AVT', 'AXTA', 'AXS', 'AXON', 'BKR', 'BALL', 'BAC', 'OZK', 'BBWI', 'BAX', 'BDX', 'BRBR', 'BSY', 'BRK B', 'BBY', 'BILL', 'BIO', 'TECH', 'BIIB', 'BMRN', 'BIRK', 'BJ', 'BLK', 'BX', 'HRB', 'XYZ', 'OWL', 'BK', 'BA', 'BOKF', 'BKNG', 'BAH', 'BWA', 'SAM

In [17]:
idx=tickers.index(True)
idx,tickers[idx]

ValueError: True is not in list

In [1]:
# trying the Russell 1000 universe
import nest_asyncio
nest_asyncio.apply()

from src.data_pipelines.run_backfill import run_backfill

run_backfill(
    universe='russell1000',
    start_date='2023-01-01',
    end_date='2026-03-09'
)

INFO:ib_insync.client:Connecting to 127.0.0.1:7497 with clientId 1...
INFO:ib_insync.client:Connected
INFO:ib_insync.client:Logged on to server version 176
INFO:ib_insync.client:API connection ready
INFO:ib_insync.wrapper:Warning 2104, reqId -1: Market data farm connection is OK:cafarm
INFO:ib_insync.wrapper:Warning 2104, reqId -1: Market data farm connection is OK:hfarm
INFO:ib_insync.wrapper:Warning 2104, reqId -1: Market data farm connection is OK:eufarmnj
INFO:ib_insync.wrapper:Warning 2104, reqId -1: Market data farm connection is OK:cashfarm
INFO:ib_insync.wrapper:Warning 2104, reqId -1: Market data farm connection is OK:usfuture
INFO:ib_insync.wrapper:Warning 2104, reqId -1: Market data farm connection is OK:afarm
INFO:ib_insync.wrapper:Warning 2104, reqId -1: Market data farm connection is OK:usopt.nj
INFO:ib_insync.wrapper:Warning 2104, reqId -1: Market data farm connection is OK:jfarm
INFO:ib_insync.wrapper:Warning 2104, reqId -1: Market data farm connection is OK:usfarm.nj
I